# 008 · Hunyuan3D 2.1 · Kaggle 2×T4

Canonical code lives in `hunyuan3d21_setup.py` and `hunyuan3d21_worker.py`; this notebook is generated by `scripts/sync_hunyuan3d21_notebook.py`.


In [ ]:
from pathlib import Path

SETUP_SOURCE = 'from __future__ import annotations\n\nimport hashlib\nimport shutil\nimport subprocess\nimport urllib.request\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\n\n@dataclass(frozen=True)\nclass Artifact:\n    name: str\n    url: str\n    sha256: str\n\n\n@dataclass(frozen=True)\nclass SetupConfig:\n    root: Path = Path("/kaggle/working/Hunyuan3D-2.1")\n    venv: Path = Path("/kaggle/working/hy21-venv")\n    wheel_dir: Path = Path("/kaggle/working/hy21-wheels")\n    commit: str = "82920d643c0dc2f7bfd7255f45f62d386edfe60c"\n    repo: str = "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git"\n\n\nCFG = SetupConfig()\nPY = CFG.venv / "bin/python"\nPIP = [str(PY), "-m", "pip"]\n\nARTIFACTS = (\n    Artifact(\n        name="custom_rasterizer-0.1-cp310-cp310-linux_x86_64.whl",\n        url=(\n            "https://github.com/xiaoqianran/kaggle-build/releases/download/"\n            "hunyuan3d21-rasterizer-t4-v1/custom_rasterizer-0.1-cp310-cp310-linux_x86_64.whl"\n        ),\n        sha256="c3dea51fdbc970c2d0c8c8718f0ef53b423383c81142532d632634aae753da80",\n    ),\n    Artifact(\n        name="mesh_inpaint_processor-0.0.1-cp310-cp310-linux_x86_64.whl",\n        url=(\n            "https://github.com/xiaoqianran/kaggle-build/releases/download/"\n            "hunyuan3d21-cp310-v1/mesh_inpaint_processor-0.0.1-cp310-cp310-linux_x86_64.whl"\n        ),\n        sha256="3572c7983dced6b21a7b018ab7e0bc885e0a26ede113402ac1c1ea5ecc515610",\n    ),\n)\n\nCORE_DEPS = [\n    "ninja==1.11.1.1",\n    "pybind11==2.13.4",\n    "transformers==4.46.0",\n    "diffusers==0.30.0",\n    "accelerate==1.1.1",\n    "pytorch-lightning==1.9.5",\n    "huggingface-hub==0.30.2",\n    "safetensors==0.4.4",\n    "numpy==1.24.4",\n    "scipy==1.14.1",\n    "einops==0.8.0",\n    "pandas==2.2.2",\n    "opencv-python==4.10.0.84",\n    "imageio==2.36.0",\n    "scikit-image==0.24.0",\n    "rembg==2.0.65",\n    "realesrgan==0.3.0",\n    "basicsr==1.4.2",\n    "trimesh==4.4.7",\n    "pymeshlab==2022.2.post3",\n    "pygltflib==1.16.3",\n    "xatlas==0.0.9",\n    "open3d==0.18.0",\n    "omegaconf==2.3.0",\n    "pyyaml==6.0.2",\n    "configargparse==1.7",\n    "tqdm==4.66.5",\n    "psutil==6.0.0",\n    "cupy-cuda12x==13.4.1",\n    "onnxruntime==1.16.3",\n    "torchmetrics==1.6.0",\n    "pydantic==2.10.6",\n    "timm",\n    "pythreejs",\n    "torchdiffeq",\n]\n\n\ndef run(args: list[str], *, cwd: Path | None = None) -> None:\n    print("+", " ".join(args), flush=True)\n    subprocess.run(args, cwd=cwd, check=True)\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef download_verified(artifact: Artifact) -> Path:\n    CFG.wheel_dir.mkdir(parents=True, exist_ok=True)\n    path = CFG.wheel_dir / artifact.name\n    if not path.exists() or sha256(path) != artifact.sha256:\n        path.unlink(missing_ok=True)\n        urllib.request.urlretrieve(artifact.url, path)\n    actual = sha256(path)\n    if actual != artifact.sha256:\n        raise RuntimeError(f"SHA256 mismatch for {artifact.name}: {actual}")\n    return path\n\n\ndef patch_once(path: Path, old: str, new: str) -> None:\n    text = path.read_text(encoding="utf-8")\n    if new in text:\n        return\n    if old not in text:\n        raise RuntimeError(f"Patch anchor not found: {path}")\n    path.write_text(text.replace(old, new, 1), encoding="utf-8")\n\n\ndef ensure_venv() -> None:\n    if PY.exists():\n        return\n    shutil.rmtree(CFG.venv, ignore_errors=True)\n    run(["uv", "venv", "--python", "3.10", "--seed", str(CFG.venv)])\n\n\ndef install_dependencies() -> None:\n    run(PIP + ["install", "-q", "--upgrade", "pip", "wheel"])\n    run(PIP + ["install", "-q", "setuptools==80.9.0"])\n    run(\n        PIP\n        + [\n            "install",\n            "-q",\n            "torch==2.5.1",\n            "torchvision==0.20.1",\n            "torchaudio==2.5.1",\n            "--index-url",\n            "https://download.pytorch.org/whl/cu124",\n        ]\n    )\n    run(PIP + ["install", "-q"] + CORE_DEPS)\n    run(PIP + ["install", "-q", "git+https://github.com/gromgull/py-vox-io.git"])\n    wheels = [str(download_verified(artifact)) for artifact in ARTIFACTS]\n    run(PIP + ["install", "-q"] + wheels)\n\n\ndef verify_runtime_imports() -> None:\n    run(\n        [\n            str(PY),\n            "-c",\n            (\n                "import pkg_resources, pytorch_lightning, torch, "\n                "custom_rasterizer, mesh_inpaint_processor; "\n                "print(\'runtime imports OK\', torch.__version__, torch.version.cuda)"\n            ),\n        ]\n    )\n\n\ndef ensure_source() -> None:\n    if not CFG.root.exists():\n        run(["git", "clone", "--filter=blob:none", CFG.repo, str(CFG.root)])\n    run(["git", "fetch", "-q", "origin", CFG.commit], cwd=CFG.root)\n    run(["git", "checkout", "-q", CFG.commit], cwd=CFG.root)\n\n\ndef patch_sources() -> None:\n    site = subprocess.check_output(\n        [str(PY), "-c", "import site; print(site.getsitepackages()[0])"], text=True\n    ).strip()\n    patch_once(\n        Path(site) / "basicsr/data/degradations.py",\n        "from torchvision.transforms.functional_tensor import rgb_to_grayscale",\n        "from torchvision.transforms.functional import rgb_to_grayscale",\n    )\n\n    multiview = CFG.root / "hy3dpaint/utils/multiview_utils.py"\n    patch_once(\n        multiview,\n        "        self.pipeline = pipeline.to(self.device)\\n",\n        "        self.pipeline = pipeline\\n"\n        "        if torch.cuda.is_available():\\n"\n        "            self.pipeline.enable_model_cpu_offload(gpu_id=0)\\n",\n    )\n    patch_once(\n        multiview,\n        "            self.dino_v2 = self.dino_v2.to(self.device)\\n",\n        \'            dino_device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"\\n\'\n        "            self.dino_v2 = self.dino_v2.to(dino_device)\\n"\n        "            self.dino_device = dino_device\\n"\n        \'            print(f"[hy21] DINO Giant resident on {dino_device}", flush=True)\\n\',\n    )\n    patch_once(\n        multiview,\n        "        kwargs = dict(generator=torch.Generator(device=self.pipeline.device).manual_seed(0))\\n",\n        \'        exec_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\\n\'\n        "        kwargs = dict(generator=torch.Generator(device=exec_device).manual_seed(0))\\n",\n    )\n    patch_once(\n        multiview,\n        \'            kwargs["dino_hidden_states"] = dino_hidden_states\\n\',\n        \'            kwargs["dino_hidden_states"] = dino_hidden_states.to("cuda:0", non_blocking=True)\\n\',\n    )\n    paint_pipeline = CFG.root / "hy3dpaint/hunyuanpaintpbr/pipeline.py"\n    patch_once(\n        paint_pipeline,\n        "        dtype = next(self.vae.parameters()).dtype\\n"\n        "        images = (images - 0.5) * 2.0\\n"\n        "        posterior = self.vae.encode(images.to(dtype)).latent_dist\\n",\n        "        dtype = next(self.vae.parameters()).dtype\\n"\n        "        device = self._execution_device\\n"\n        "        images = (images - 0.5) * 2.0\\n"\n        "        images = images.to(device=device, dtype=dtype, non_blocking=True)\\n"\n        "        posterior = self.vae.encode(images).latent_dist\\n",\n    )\n\n    paint_model = CFG.root / "hy3dpaint/hunyuanpaintpbr/unet/model.py"\n    patch_once(\n        paint_model,\n        "        dtype = next(self.pipeline.vae.parameters()).dtype\\n\\n"\n        "        images = (images - 0.5) * 2.0\\n"\n        "        posterior = self.pipeline.vae.encode(images.to(dtype)).latent_dist\\n",\n        "        dtype = next(self.pipeline.vae.parameters()).dtype\\n"\n        "        device = self.pipeline._execution_device\\n\\n"\n        "        images = (images - 0.5) * 2.0\\n"\n        "        images = images.to(device=device, dtype=dtype, non_blocking=True)\\n"\n        "        posterior = self.pipeline.vae.encode(images).latent_dist\\n",\n    )\n\n    patch_once(\n        CFG.root / "hy3dpaint/DifferentiableRenderer/mesh_utils.py",\n        "import bpy\\n",\n        "try:\\n    import bpy\\nexcept ImportError:\\n    bpy = None\\n",\n    )\n\n\ndef install_mesh_extension() -> None:\n    module_path = subprocess.check_output(\n        [\n            str(PY),\n            "-c",\n            "import mesh_inpaint_processor; print(mesh_inpaint_processor.__file__)",\n        ],\n        text=True,\n    ).strip()\n    target = CFG.root / "hy3dpaint/DifferentiableRenderer" / Path(module_path).name\n    shutil.copy2(module_path, target)\n\n\ndef ensure_realesrgan() -> None:\n    path = CFG.root / "hy3dpaint/ckpt/RealESRGAN_x4plus.pth"\n    if path.exists():\n        return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    run(\n        [\n            "wget",\n            "-q",\n            "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",\n            "-O",\n            str(path),\n        ]\n    )\n\n\ndef main() -> None:\n    ensure_venv()\n    install_dependencies()\n    verify_runtime_imports()\n    ensure_source()\n    patch_sources()\n    install_mesh_extension()\n    ensure_realesrgan()\n    run(\n        [\n            str(PY),\n            "-c",\n            "import torch; print(torch.__version__, torch.cuda.device_count())",\n        ]\n    )\n    print("✅ Hunyuan3D 2.1 environment ready", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
WORKER_SOURCE = 'from __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport os\nimport socket\nimport sys\nimport tempfile\nimport threading\nimport time\nimport traceback\nimport uuid\nfrom dataclasses import dataclass, replace\nfrom pathlib import Path\nfrom typing import Any\nfrom urllib.parse import urljoin\n\nimport requests\nimport torch\nfrom cryptography.hazmat.primitives.ciphers.aead import AESGCM\n\n\n@dataclass(frozen=True)\nclass HunyuanConfig:\n    root: Path = Path("/kaggle/working/Hunyuan3D-2.1")\n    image: Path = Path("/kaggle/working/Hunyuan3D-2.1/assets/demo.png")\n    workdir: Path = Path("/kaggle/working")\n    shape_steps: int = 20\n    octree_resolution: int = 256\n    paint_views: int = 4\n    paint_resolution: int = 256\n    render_size: int = 1024\n    texture_size: int = 2048\n\n    @property\n    def shape_obj(self) -> Path:\n        return self.workdir / "hunyuan21-shape.obj"\n\n    @property\n    def shape_glb(self) -> Path:\n        return self.workdir / "hunyuan21-shape.glb"\n\n    @property\n    def pbr_obj(self) -> Path:\n        return self.workdir / "hunyuan21-pbr.obj"\n\n    @property\n    def pbr_glb(self) -> Path:\n        return self.workdir / "hunyuan21-pbr.glb"\n\n\nCFG = HunyuanConfig()\n\n\ndef prepare_imports() -> None:\n    sys.path[:0] = [str(CFG.root / "hy3dshape"), str(CFG.root / "hy3dpaint")]\n    os.chdir(CFG.root)\n\n\ndef clear_cuda() -> None:\n    gc.collect()\n    for index in range(torch.cuda.device_count()):\n        with torch.cuda.device(index):\n            torch.cuda.empty_cache()\n\n\ndef reset_peaks() -> None:\n    for index in range(torch.cuda.device_count()):\n        with torch.cuda.device(index):\n            torch.cuda.reset_peak_memory_stats()\n\n\ndef memory_snapshot(label: str) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    for index in range(torch.cuda.device_count()):\n        with torch.cuda.device(index):\n            free, _ = torch.cuda.mem_get_info()\n            rows.append(\n                {\n                    "gpu": index,\n                    "free_gib": round(free / 2**30, 2),\n                    "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 2),\n                    "peak_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),\n                }\n            )\n    print(f"[HY21_MEM] {label} {json.dumps(rows)}", flush=True)\n    return rows\n\n\ndef run_shape(cfg: HunyuanConfig = CFG) -> float:\n    from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline\n\n    started = time.perf_counter()\n    pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(\n        "tencent/Hunyuan3D-2.1",\n        subfolder="hunyuan3d-dit-v2-1",\n        use_safetensors=False,\n        torch_dtype=torch.float16,\n    )\n    pipeline.to("cuda:0")\n    if not callable(pipeline):\n        raise TypeError("Hunyuan shape pipeline is not callable after device placement")\n    print(f"[HY21] shape_load={time.perf_counter() - started:.2f}s", flush=True)\n    memory_snapshot("shape-loaded")\n\n    started = time.perf_counter()\n    with torch.inference_mode():\n        mesh = pipeline(\n            image=str(cfg.image),\n            num_inference_steps=cfg.shape_steps,\n            octree_resolution=cfg.octree_resolution,\n            guidance_scale=5.0,\n        )[0]\n    seconds = time.perf_counter() - started\n    mesh.export(cfg.shape_obj)\n    mesh.export(cfg.shape_glb)\n    print(\n        f"[HY21_SHAPE] seconds={seconds:.3f} glb={cfg.shape_glb.stat().st_size}",\n        flush=True,\n    )\n\n    del mesh, pipeline\n    clear_cuda()\n    memory_snapshot("shape-released")\n    return seconds\n\n\ndef run_paint(cfg: HunyuanConfig = CFG) -> float:\n    from textureGenPipeline import Hunyuan3DPaintConfig, Hunyuan3DPaintPipeline\n\n    config = Hunyuan3DPaintConfig(\n        max_num_view=cfg.paint_views, resolution=cfg.paint_resolution\n    )\n    config.render_size = cfg.render_size\n    config.texture_size = cfg.texture_size\n    realesrgan = cfg.root / "hy3dpaint/ckpt/RealESRGAN_x4plus.pth"\n    if not realesrgan.is_file():\n        raise FileNotFoundError(realesrgan)\n    config.realesrgan_ckpt_path = str(realesrgan)\n\n    started = time.perf_counter()\n    pipeline = Hunyuan3DPaintPipeline(config)\n    print(f"[HY21] paint_load={time.perf_counter() - started:.2f}s", flush=True)\n    memory_snapshot("paint-loaded")\n\n    started = time.perf_counter()\n    pipeline(\n        mesh_path=str(cfg.shape_obj),\n        image_path=str(cfg.image),\n        output_mesh_path=str(cfg.pbr_obj),\n        use_remesh=True,\n        save_glb=False,\n    )\n    seconds = time.perf_counter() - started\n    if not cfg.pbr_obj.is_file():\n        raise FileNotFoundError(cfg.pbr_obj)\n    print(\n        f"[HY21_PAINT] seconds={seconds:.3f} obj={cfg.pbr_obj.stat().st_size}",\n        flush=True,\n    )\n    memory_snapshot("paint-finished")\n    return seconds\n\n\ndef export_pbr_glb(cfg: HunyuanConfig = CFG) -> None:\n    from convert_utils import create_glb_with_pbr_materials\n\n    base = cfg.pbr_obj.with_suffix("")\n    assets = {\n        "obj": cfg.pbr_obj,\n        "mtl": base.with_suffix(".mtl"),\n        "albedo": Path(f"{base}.jpg"),\n        "metallic": Path(f"{base}_metallic.jpg"),\n        "roughness": Path(f"{base}_roughness.jpg"),\n    }\n    missing = [str(path) for path in assets.values() if not path.is_file()]\n    if missing:\n        raise FileNotFoundError(f"Missing PBR assets: {missing}")\n\n    textures = {name: str(assets[name]) for name in ("albedo", "metallic", "roughness")}\n    create_glb_with_pbr_materials(str(cfg.pbr_obj), textures, str(cfg.pbr_glb))\n    if not cfg.pbr_glb.is_file() or cfg.pbr_glb.stat().st_size < 1000:\n        raise RuntimeError(\n            "PBR GLB export failed or produced an unexpectedly small file"\n        )\n    print(f"[HY21_EXPORT] glb={cfg.pbr_glb.stat().st_size}", flush=True)\n\n\nMODEL = "hunyuan3d-2.1"\nPOLL_TIMEOUT = 35\nREQUEST_TIMEOUT = 180\nHEARTBEAT_SECONDS = 10\n\n\ndef encrypt_blob(data: bytes, token: str) -> bytes:\n    key = hashlib.sha256(token.encode()).digest()\n    nonce = os.urandom(12)\n    return nonce + AESGCM(key).encrypt(nonce, data, None)\n\n\ndef api_url(base_url: str, path: str) -> str:\n    return urljoin(base_url.rstrip("/") + "/", path.lstrip("/"))\n\n\ndef auth_headers(token: str) -> dict[str, str]:\n    return {\n        "Authorization": f"Bearer {token}",\n        "Cache-Control": "no-cache, no-store",\n        "Pragma": "no-cache",\n    }\n\n\ndef checked_response(response: requests.Response, label: str) -> requests.Response:\n    if response.status_code >= 400:\n        body = response.text[:1000].replace("\\n", " ")\n        raise RuntimeError(f"{label}: HTTP {response.status_code} | {body}")\n    return response\n\n\ndef preflight_hub(base_url: str, token: str, session: requests.Session) -> str:\n    response = session.get(api_url(base_url, "/api/models"), params={"_ts": time.time_ns()}, timeout=20)\n    checked_response(response, "GET /api/models")\n    model_ids = {item.get("id") for item in response.json() if isinstance(item, dict)}\n    if MODEL not in model_ids:\n        raise RuntimeError(f"Hub does not advertise {MODEL}. Update/restart Hub first.")\n    checked_response(\n        session.get(api_url(base_url, "/api/failed"), params={"_ts": time.time_ns()}, timeout=20),\n        "GET /api/failed (auth check)",\n    )\n    snapshot = session.get(api_url(base_url, "/api/status"), params={"_ts": time.time_ns()}, timeout=20)\n    checked_response(snapshot, "GET /api/status")\n    payload = snapshot.json()\n    instance_id = str(payload.get("hub_instance_id") or "")\n    if payload.get("storage") != "sqlite" or not instance_id:\n        raise RuntimeError("Hub protocol is too old; SQLite state + hub_instance_id are required")\n    print(\n        f"[preflight] Hub OK | instance={instance_id[:12]} | "\n        f"queued={payload.get(\'queued_by_model\', {}).get(MODEL, 0)} "\n        f"inflight={payload.get(\'inflight_by_model\', {}).get(MODEL, 0)}",\n        flush=True,\n    )\n    return instance_id\n\n\ndef heartbeat_loop(\n    base_url: str, token: str, worker_id: str, stop: threading.Event, active_task: dict[str, int | None]\n) -> None:\n    session = requests.Session()\n    session.headers.update(auth_headers(token))\n    while not stop.wait(HEARTBEAT_SECONDS):\n        try:\n            session.post(\n                api_url(base_url, "/worker/heartbeat"),\n                json={\n                    "worker_id": worker_id,\n                    "local_queue": 0,\n                    "upload_queue": 0,\n                    "active_task_id": active_task["id"],\n                    "meta": {"persistent": True, "dual_t4": True},\n                },\n                timeout=15,\n            ).raise_for_status()\n        except Exception as exc:\n            print(f"[heartbeat] {type(exc).__name__}: {exc}", flush=True)\n\n\ndef report_failure(\n    base_url: str, session: requests.Session, task_id: int, exc: BaseException\n) -> None:\n    message = f"{type(exc).__name__}: {exc}"\n    print(f"[HY21] FAIL #{task_id}: {message}", flush=True)\n    try:\n        session.post(\n            api_url(base_url, "/task/fail"),\n            json={"id": task_id, "error": message[:1900], "requeue": True},\n            timeout=20,\n        ).raise_for_status()\n    except Exception as report_exc:\n        print(f"[HY21] fail-report error: {report_exc}", flush=True)\n\n\ndef run_hub_task(\n    base_url: str, token: str, session: requests.Session, worker_id: str, task: dict[str, Any]\n) -> None:\n    task_id = int(task["id"])\n    started = time.perf_counter()\n    response = session.get(api_url(base_url, task["input_url"]), timeout=60)\n    checked_response(response, "GET task input")\n    suffix = Path(str(task.get("source_label") or "input.png")).suffix.lower()\n    if suffix not in {".png", ".jpg", ".jpeg", ".webp"}:\n        suffix = ".png"\n\n    with tempfile.TemporaryDirectory(prefix=f"hunyuan3d21-{task_id}-") as tmp:\n        workdir = Path(tmp)\n        image_path = workdir / f"input{suffix}"\n        image_path.write_bytes(response.content)\n        cfg = replace(\n            CFG,\n            image=image_path,\n            workdir=workdir,\n            shape_steps=int(task.get("shape_steps", CFG.shape_steps)),\n            octree_resolution=int(task.get("octree_resolution", CFG.octree_resolution)),\n            paint_views=int(task.get("paint_views", CFG.paint_views)),\n            paint_resolution=int(task.get("paint_resolution", CFG.paint_resolution)),\n            texture_size=int(task.get("texture_size", CFG.texture_size)),\n        )\n        reset_peaks()\n        shape_seconds = run_shape(cfg)\n        paint_seconds = run_paint(cfg)\n        export_pbr_glb(cfg)\n        artifact = cfg.pbr_glb.read_bytes()\n        encrypted = encrypt_blob(artifact, token)\n        elapsed = time.perf_counter() - started\n        upload = session.post(\n            api_url(base_url, "/upload/artifact"),\n            data={\n                "id": str(task_id),\n                "model": MODEL,\n                "worker_id": worker_id,\n                "gpu": "0",\n                "seconds": f"{elapsed:.3f}",\n                "output_format": "glb",\n            },\n            files={"file": (f"{task_id}.glb.bin", encrypted, "application/octet-stream")},\n            timeout=REQUEST_TIMEOUT,\n        )\n        checked_response(upload, "POST /upload/artifact")\n        print(\n            f"[HY21] ✓ #{task_id} total={elapsed:.2f}s shape={shape_seconds:.2f}s "\n            f"paint={paint_seconds:.2f}s glb={len(artifact)}",\n            flush=True,\n        )\n\n\ndef hub_main() -> None:\n    prepare_imports()\n    if torch.cuda.device_count() < 2:\n        raise RuntimeError(f"2 GPUs required, got {torch.cuda.device_count()}")\n    base_url = os.environ["BASE_URL"].rstrip("/")\n    token = os.environ["KAGGLE_HUB_TOKEN"]\n    run_id = f"{socket.gethostname()[:8]}-{uuid.uuid4().hex[:6]}"\n    worker_id = f"hunyuan3d21-{run_id}"\n    gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]\n    session = requests.Session()\n    session.headers.update(auth_headers(token))\n    hub_instance_id = preflight_hub(base_url, token, session)\n    register = session.post(\n        api_url(base_url, "/worker/register"),\n        json={\n            "worker_id": worker_id,\n            "model": MODEL,\n            "gpus": gpu_names,\n            "runtime": "hunyuan3d21-persistent-py310",\n            "concurrency": 1,\n            "meta": {"torch": torch.__version__, "torch_cuda": torch.version.cuda, "persistent": True},\n        },\n        timeout=30,\n    )\n    checked_response(register, "POST /worker/register")\n    print(f"[HY21] registered {worker_id} | GPUs={gpu_names}", flush=True)\n\n    stop = threading.Event()\n    active_task: dict[str, int | None] = {"id": None}\n    threading.Thread(\n        target=heartbeat_loop,\n        args=(base_url, token, worker_id, stop, active_task),\n        daemon=True,\n    ).start()\n    try:\n        while True:\n            task: dict[str, Any] | None = None\n            try:\n                response = session.post(\n                    api_url(base_url, "/task/claim"),\n                    json={"model": MODEL, "worker_id": worker_id, "wait_seconds": 25},\n                    timeout=POLL_TIMEOUT,\n                )\n                response_instance = response.headers.get("X-Hub-Instance", "")\n                if response_instance and response_instance != hub_instance_id:\n                    raise RuntimeError(\n                        f"Hub instance changed: expected={hub_instance_id[:12]} got={response_instance[:12]}"\n                    )\n                if response.status_code == 204:\n                    continue\n                checked_response(response, "POST /task/claim")\n                task = response.json()\n                active_task["id"] = int(task["id"])\n                print(f"[HY21] ↓ #{task[\'id\']} {task.get(\'source_label\', \'input\')}", flush=True)\n                run_hub_task(base_url, token, session, worker_id, task)\n                active_task["id"] = None\n            except KeyboardInterrupt:\n                raise\n            except Exception as exc:\n                if task is not None and "id" in task:\n                    report_failure(base_url, session, int(task["id"]), exc)\n                    active_task["id"] = None\n                else:\n                    print(f"[HY21] poll error: {type(exc).__name__}: {exc}", flush=True)\n                    time.sleep(2)\n                traceback.print_exc()\n    except KeyboardInterrupt:\n        pass\n    finally:\n        stop.set()\n\n\ndef benchmark_main() -> None:\n    prepare_imports()\n    if torch.cuda.device_count() < 2:\n        raise RuntimeError(f"2 GPUs required, got {torch.cuda.device_count()}")\n    reset_peaks()\n    shape_seconds = run_shape()\n    paint_seconds = run_paint()\n    export_pbr_glb()\n    summary = {\n        "shape_seconds": round(shape_seconds, 3),\n        "paint_seconds": round(paint_seconds, 3),\n        "shape_glb_bytes": CFG.shape_glb.stat().st_size,\n        "pbr_glb_bytes": CFG.pbr_glb.stat().st_size,\n    }\n    (CFG.workdir / "hunyuan21-benchmark.json").write_text(\n        json.dumps(summary, indent=2), encoding="utf-8"\n    )\n    print(f"[HY21_RESULT] {json.dumps(summary)}", flush=True)\n    print("✅ Hunyuan3D 2.1 Shape + Paint 完整自检通过", flush=True)\n\n\ndef main() -> None:\n    if os.getenv("HUNYUAN3D21_HUB_MODE", "").strip().lower() in {"1", "true", "yes", "on"}:\n        hub_main()\n    else:\n        benchmark_main()\n\n\nif __name__ == "__main__":\n    main()\n'

Path('/kaggle/working/hunyuan3d21_setup.py').write_text(SETUP_SOURCE, encoding='utf-8')
Path('/kaggle/working/hunyuan3d21_worker.py').write_text(WORKER_SOURCE, encoding='utf-8')
print('✅ Hunyuan3D 2.1 helper modules materialized')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '/kaggle/working/hunyuan3d21_setup.py'], check=True)


In [ ]:
import subprocess
subprocess.run([
    '/kaggle/working/hy21-venv/bin/python',
    '/kaggle/working/hunyuan3d21_worker.py',
], check=True, timeout=2700)


In [ ]:
import os, subprocess
from pathlib import Path
BASE_URL = os.getenv('KAGGLE_HUB_BASE_URL', 'https://ranran-sana.202820.xyz').rstrip('/')
TOKEN = os.getenv('KAGGLE_HUB_TOKEN', '')
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret('KAGGLE_HUB_TOKEN') or ''
    except Exception:
        TOKEN = ''
if not TOKEN:
    print('ℹ️ KAGGLE_HUB_TOKEN 未配置：Hunyuan3D 2.1 自检已完成，跳过常驻 Hub Worker。')
else:
    log = Path('/kaggle/working/hunyuan3d21-worker.log')
    pid_file = Path('/kaggle/working/hunyuan3d21-worker.pid')
    env = os.environ.copy()
    env.update({'BASE_URL': BASE_URL, 'KAGGLE_HUB_TOKEN': TOKEN, 'HUNYUAN3D21_HUB_MODE': '1', 'PYTHONUNBUFFERED': '1'})
    handle = log.open('ab', buffering=0)
    process = subprocess.Popen(['/kaggle/working/hy21-venv/bin/python', '/kaggle/working/hunyuan3d21_worker.py'], env=env, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
    pid_file.write_text(str(process.pid))
    print('✅ Hunyuan3D 2.1 Hub worker started | PID', process.pid, '| log', log)


## 验收

成功必须出现 `[HY21_SHAPE]`、`[HY21_PAINT]`、`[HY21_EXPORT]`、`[HY21_RESULT]` 和最终成功标记。
